# Train a decision model on title/abstract screening (Unsloth)

Fine-tunes a decision model (default: Laya multilingual; Qwen3.5 as an option) with [Unsloth](https://unsloth.ai/docs/basics/train-your-own-decision-model-with-unsloth)
and evaluates it on the validation reviews at fixed step intervals.

| | |
|---|---|
| Data | `data/processed/training/requests/<config>/`, built by [scripts/build_requests.py](../../scripts/build_requests.py) (see README, *Training data*) |
| Input | review: selection criteria; record: title + abstract (one `noul` question `include`) |
| Train | SYNERGY+ train split, without papers that also occur in val/test (leakage guard of the build script) |
| Validation | 17 CLEF TAR 2019 Intervention training reviews (not the test reviews) |
| Labels | train: title/abstract decision where the dataset has one, else after full text. Val metrics on both CLEF TAR levels: **abstract-level** and **content-level** (after full text) |
| What is trained | LoRA adapters on the backbone only. Laya's pretrained decision head stays frozen (`TRAIN_HEAD`); a Qwen model needs its new head trained. The backbone weights stay frozen. |
| GPU | **only physical GPU 1** (first code cell; GPU 0 is never visible to this kernel) |

Validation metrics, macro over reviews as in the Jev experiments: MAP, WSS@95%, TNR@95%, R@10%. The model is
chosen by abstract-level MAP. Every validation score is written to `runs/<name>/val_scores/` so metrics can be
recomputed without the model.

**Pilot first:** set `PILOT = True` for a 300-step run that tests the whole pipeline in minutes.

**Step 0:** before training, the untrained model is evaluated on val (`val_scores/step-000000.jsonl`), so every
later step can be compared with where the model started.

**Balanced train data:** `BALANCED = True` trains on the requests built with
`python scripts/build_requests.py --train-neg-per-pos 1` (per review, one random exclude per include; 27k instead
of 267k requests). Val is the same file either way. Balanced runs use more epochs because one epoch is about 420 steps.

**Model:** Laya multilingual (mmBERT-base encoder, about 300M parameters, already trained on decisions, positions up to 8,192 tokens; its own training used 1,024). Qwen3.5-0.8B / 4B (a new decision head on the language model) are in `MODELS`.

Run headless on the server (survives a closed connection):
```bash
tmux new -s train
source .venv/bin/activate
jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 \
    notebooks/training/train_decision_model.ipynb --output-dir runs/notebooks
```

In [1]:
# GPU 1 only. Must run before torch / unsloth are imported (restart the kernel if they already are).
import os, sys
assert "torch" not in sys.modules, "Restart the kernel: torch was imported before the GPU was chosen."
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # number GPUs as nvidia-smi does
os.environ["CUDA_VISIBLE_DEVICES"] = "1"         # physical GPU 1; GPU 0 is never used

import unsloth  # noqa: F401  (Unsloth patches transformers, so it is imported first)
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1, "expected exactly one visible GPU (physical GPU 1)"
print("training on:", torch.cuda.get_device_name(0), "= physical GPU", os.environ["CUDA_VISIBLE_DEVICES"])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/joshua/decision-model-screener/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1118 14:14:09.502000 899626 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1118 14:14:09.518000 899626 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
training on: NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition = physical GPU 1


## Settings

In [2]:
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
DATA = ROOT / "data" / "processed" / "training"
BALANCED = True   # True: train on 1 exclude per include (build with --train-neg-per-pos 1); val is unchanged
REQUESTS = "review-criteria__record-title+abstract__label-abstract_else_final" + ("__train-neg1" if BALANCED else "")   # output of scripts/build_requests.py
REQ_DIR = DATA / "requests" / REQUESTS

PILOT = False   # True: 300 steps, to test the pipeline

# Pinned Hub revisions (never an unpinned "latest" for a reported result)
MODELS = {"convaiinnovations/laya-multilingual": "92d22143468d8af669dc1a5a0480a2b21b4fe56e",
          "unsloth/Qwen3.5-0.8B": "23c69c53358a07516b5827588b3fdb12ae78fd65",
          "unsloth/Qwen3.5-4B": "3764fa359b9082ea5a1e4a5e3ac3aaf6e9671636"}
MODEL = "convaiinnovations/laya-multilingual"
REVISION = MODELS[MODEL]

MAX_SEQ_LENGTH = 8192    # tokens (Laya's position limit); checked below: no val input is cut, train cuts are reported
LORA_R, LORA_ALPHA = 16, 16
LR, HEAD_LR = 2e-4, 1e-4  # LoRA adapters; the decision head (Unsloth's default)
TRAIN_HEAD = False        # Laya: keep its pretrained head frozen (adapters only). A Qwen model always trains its new head.
BATCH, ACCUM = 64, 1      # effective batch 64
EPOCHS = 3 if BALANCED else 1   # one epoch: about 4,200 steps full, 420 steps balanced
MAX_STEPS = 1000 if PILOT else -1    # -1 = full epochs
EVAL_STEPS = 500 if PILOT else 1000  # validation every N optimizer steps (one pass over val takes about 4 min)
EVAL_BATCH = 128
SEED = 3407

RUN = ROOT / "runs" / f"{MODEL.split('/')[-1]}__{REQUESTS}{'__pilot' if PILOT else ''}"
RUN.mkdir(parents=True, exist_ok=True)
print(RUN)

/home/joshua/decision-model-screener/runs/laya-multilingual__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1


## Data and leakage check

Train and val must not share a review, a review title or a paper (PMID, DOI or title, the same rule as the
build script). The check runs on the files that are actually trained on.

In [3]:
import json
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT / "scripts"))
from build_requests import paper_keys, norm_title   # same matching rule as the build
from screening import metrics                        # same metric code as the Jev experiments

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

build_config = json.loads((REQ_DIR / "config.json").read_text())
train_rows, train_meta = read_jsonl(REQ_DIR / "train.jsonl"), read_jsonl(REQ_DIR / "train.meta.jsonl")
val_rows, val_meta = read_jsonl(REQ_DIR / "val.jsonl"), read_jsonl(REQ_DIR / "val.meta.jsonl")
reviews = {r["review_id"]: r for r in read_jsonl(DATA / "reviews.jsonl")}

needed = {(m["review_id"], m["record_id"]) for m in train_meta + val_meta}
records = {}
with open(DATA / "records.jsonl") as f:
    for line in f:
        r = json.loads(line)
        if (r["review_id"], r["record_id"]) in needed:
            records[r["review_id"], r["record_id"]] = r

train_reviews = {m["review_id"] for m in train_meta}
val_reviews = {m["review_id"] for m in val_meta}
assert not train_reviews & val_reviews, "review in train and val"
assert not ({norm_title(reviews[r]["title"]) for r in train_reviews}
            & {norm_title(reviews[r]["title"]) for r in val_reviews}), "same review title in train and val"
train_keys = set()
for m in train_meta:
    train_keys |= paper_keys(records[m["review_id"], m["record_id"]])
shared = [m for m in val_meta if paper_keys(records[m["review_id"], m["record_id"]]) & train_keys]
assert not shared, f"{len(shared)} val records are the same paper as a train record"
print(f"no overlap: {len(train_reviews)} train reviews / {len(train_rows):,} requests, "
      f"{len(val_reviews)} val reviews / {len(val_rows):,} requests")
print("train reviews dropped by the build for overlap:", build_config["train_reviews_dropped_for_overlap"])
print("train records removed as same paper as val/test:", build_config["train_records_dropped_same_paper_as_val_test"])

# Both CLEF TAR label levels for the val metrics (the request file carries the abstract-level one)
val_labels = {lvl: [records[m["review_id"], m["record_id"]][f"label_{lvl}"] for m in val_meta]
              for lvl in ("abstract", "final")}
del records

no overlap: 92 train reviews / 27,004 requests, 17 val reviews / 27,870 requests
train reviews dropped by the build for overlap: {'Brouwer_2019': '21% of CD010558', 'Kerschbaumer_2024a': 'same review title as Kerschbaumer_2024b', 'Parodis_2023a': 'same review title as Parodis_2023b', 'Webers_2022': '40% of Kerschbaumer_2024b'}
train records removed as same paper as val/test: 6431


In [4]:
# Request format (label on the question) -> Unsloth format (answers in "gold")
def to_unsloth(row):
    questions = {name: {k: v for k, v in q.items() if k != "label"} for name, q in row["questions"].items()}
    return {"state": row["state"], "questions": questions,
            "gold": {name: q["label"] for name, q in row["questions"].items()}}

train_data = [to_unsloth(r) for r in train_rows]
val_data = [to_unsloth(r) for r in val_rows]
print(json.dumps(val_data[0], indent=1, ensure_ascii=False)[:1500])

{
 "state": {
  "review": {
   "criteria": "Types of studies: We included only RCTs in this review.\nTypes of participants: We included trials in which participants had neovascular AMD as defined by study investigators.\nTypes of interventions: We included studies that compared anti‐VEGF treatment versus another treatment, sham treatment, or no treatment. We did not include studies that compared different doses of one anti‐VEGF treatment against another, studies that included no control or comparator group, or studies that used anti‐VEGF agents in combination with other treatments. We did not include studies of aflibercept (VEGF Trap‐Eye/EYLEA solution) or studies that compared different treatment schedules (e.g. monthly vs as needed dosing), because other Cochrane reviews have evaluated these interventions (Li 2016; Sarwar 2016).\nTypes of outcome measures: Primary outcomes The primary outcome for this review was based on best‐corrected visual acuity (BCVA) at one‐year follow‐up. All 

## Model: LoRA adapters only

In [5]:
from unsloth import FastDecisionModel, DecisionTrainer

model, tokenizer = FastDecisionModel.from_pretrained(
    model_name=MODEL,
    revision=REVISION,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,     # 16-bit backbone, bf16 like Laya's own training
    load_in_4bit=False,
    random_state=SEED,
)
model = FastDecisionModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    random_state=SEED,
)

# The backbone is model.encoder (Laya's mmBERT or Qwen); everything else is the decision head.
head_trains = TRAIN_HEAD or getattr(model, "is_clef", False)   # a Qwen head is new and must train
for n, p in model.named_parameters():
    if not n.startswith("encoder."):
        p.requires_grad_(head_trains)
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
backbone = [n for n in trainable if n.startswith("encoder.") and "lora_" not in n]
assert not backbone, f"backbone weights would train: {backbone[:5]}"
assert any("lora_" in n for n in trainable), "no LoRA adapters are trainable"
print("head trains:", head_trains)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"trainable {n_train / 1e6:.1f}M of {n_all / 1e6:.0f}M parameters")

# Position of the "yes" option in the logits: Laya orders noul options (false, true), Clef-style heads (true, false)
YES = 0 if getattr(model, "is_clef", False) else 1
print("max_len", model.decision_config["max_len"], "| yes option at index", YES)

/home/joshua/decision-model-screener/.venv/lib/python3.14/site-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


head trains: False
trainable 3.4M of 325M parameters
max_len 8192 | yes option at index 1


In [6]:
train_items, train_report = FastDecisionModel.build_dataset(train_data, tokenizer, model)
val_items, val_report = FastDecisionModel.build_dataset(val_data, tokenizer, model)
print("train:", train_report)
print("val:  ", val_report)

# Never drop records; no val input may be cut (its abstract would be scored on less text).
assert train_report["skipped"] == 0 and val_report["skipped"] == 0, "records were skipped"
assert [it["row"] for it in val_items] == list(range(len(val_rows))), "val order changed"
assert val_report["truncated"] == 0, f"{val_report['truncated']} val inputs exceed MAX_SEQ_LENGTH: raise it"
lengths = sorted(len(it["input_ids"]) for it in val_items)
print("val tokens: median", lengths[len(lengths) // 2], "max", lengths[-1])

Unsloth: 1 of 27,004 training inputs are longer than max_seq_length = 8192, so the end of their state is cut. Raise max_seq_length to train on all of it.
train: {'total': 27004, 'skipped': 0, 'reason': None, 'truncated': 1}
val:   {'total': 27870, 'skipped': 0, 'reason': None, 'truncated': 0}
val tokens: median 885 max 3547


## Validation metrics at each evaluation

Ranking per review by P(include), ties broken by record id. Reviews without includes at a label level are skipped
for that level (as in the Jev experiments).

In [7]:
import numpy as np
import pandas as pd

def ranking_metrics(p_include):
    df = pd.DataFrame({"review_id": [m["review_id"] for m in val_meta],
                       "record_id": [m["record_id"] for m in val_meta],
                       "p": p_include, **{f"label_{k}": v for k, v in val_labels.items()}})
    out = {}
    for level in ("abstract", "final"):
        per_review = []
        for _, g in df.groupby("review_id"):
            ranked = g.sort_values(["p", "record_id"], ascending=[False, True])
            m = metrics(ranked[f"label_{level}"].astype(int))
            if m:
                per_review.append(m)
        mean = pd.DataFrame(per_review).mean() * 100
        for name, key in (("MAP", "map"), ("WSS@95%", "wss95"), ("TNR@95%", "tnr95"), ("R@10%", "r10")):
            out[f"{key}_{level}"] = round(float(mean[name]), 2)
        out[f"reviews_{level}"] = len(per_review)
    return df, out

trainer = None   # set below; compute_metrics reads the current step from it

def compute_metrics(pred):
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    logits = torch.tensor(np.asarray(logits)[:, :2], dtype=torch.float32)
    p_include = torch.softmax(logits, -1)[:, YES].numpy()
    assert len(p_include) == len(val_meta)
    df, out = ranking_metrics(p_include)
    step = trainer.state.global_step if trainer is not None else 0
    (RUN / "val_scores").mkdir(parents=True, exist_ok=True)   # here, not at definition: RUN may change (PILOT)
    df.to_json(RUN / "val_scores" / f"step-{step:06d}.jsonl", orient="records", lines=True)
    with open(RUN / "val_metrics.jsonl", "a") as f:
        f.write(json.dumps({"step": step, **out}) + "\n")
    return out

## Training

In [8]:
from transformers import TrainingArguments
import subprocess

args = TrainingArguments(
    output_dir=str(RUN / "checkpoints"),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=ACCUM,
    per_device_eval_batch_size=EVAL_BATCH,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    #warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,              # one checkpoint (adapters + head) per evaluation
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="map_abstract",
    greater_is_better=True,
    seed=SEED,
    report_to="none",
)
trainer = DecisionTrainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=train_items,
    eval_dataset=val_items,
    compute_metrics=compute_metrics,
    head_learning_rate=HEAD_LR,
)

# What this run used, next to its results
(RUN / "run_config.json").write_text(json.dumps({
    "model": MODEL, "revision": REVISION, "requests": REQUESTS, "build_config": build_config,
    "max_seq_length": MAX_SEQ_LENGTH, "train_head": head_trains, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lr": LR, "head_lr": HEAD_LR,
    "batch": BATCH, "accum": ACCUM, "epochs": EPOCHS, "max_steps": MAX_STEPS, "eval_steps": EVAL_STEPS,
    "balanced": BALANCED, "seed": SEED, "gpu": "physical GPU " + os.environ["CUDA_VISIBLE_DEVICES"],
    "train_report": train_report, "val_report": val_report}, indent=2, default=str))
(RUN / "pip_freeze.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                                   capture_output=True, text=True).stdout)

# Step 0: the untrained model on val (writes val_scores/step-000000.jsonl and a row in val_metrics.jsonl).
# Skip it when resuming, it is already on disk.
step0 = trainer.evaluate()
print({k: v for k, v in step0.items() if k.startswith("eval_map")})

# Resume after an interruption: trainer.train(resume_from_checkpoint=True)
train_result = trainer.train()
print(train_result)

Training Loss,Validation Loss,Step,Map Abstract,Wss95 Abstract,Tnr95 Abstract,R10 Abstract,Reviews Abstract,Map Final,Wss95 Final,Tnr95 Final,R10 Final,Reviews Final
No log,1.428922,0,7.470000,13.040000,18.450000,19.000000,17,5.760000,19.920000,25.190000,22.690000,16


{'eval_map_abstract': 7.47, 'eval_map_final': 5.76}
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Map Abstract,Wss95 Abstract,Tnr95 Abstract,R10 Abstract,Reviews Abstract,Map Final,Wss95 Final,Tnr95 Final,R10 Final,Reviews Final
1000,0.324816,0.456314,23.630000,46.300000,53.520000,50.340000,17,24.400000,55.790000,62.260000,57.800000,16
1266,0.310524,0.484062,24.160000,47.040000,54.260000,50.240000,17,24.990000,55.160000,61.610000,58.070000,16


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
TrainOutput(global_step=1266, training_loss=0.436251416308055, metrics={'train_runtime': 3037.5675, 'train_samples_per_second': 26.67, 'train_steps_per_second': 0.417, 'total_flos': 0.0, 'train_loss': 0.436251416308055, 'epoch': 3.0})


## Result

In [9]:
history = pd.read_json(RUN / "val_metrics.jsonl", lines=True).drop_duplicates("step", keep="last").set_index("step")
print("best checkpoint:", trainer.state.best_model_checkpoint)
history

best checkpoint: /home/joshua/decision-model-screener/runs/laya-multilingual__review-criteria__record-title+abstract__label-abstract_else_final__train-neg1/checkpoints/checkpoint-1266


,map_abstract,wss95_abstract,tnr95_abstract,r10_abstract,reviews_abstract,map_final,wss95_final,tnr95_final,r10_final,reviews_final
step,,,,,,,,,,
0,7.47,13.04,18.45,19.00,17,5.76,19.92,25.19,22.69,16
1000,23.63,46.30,53.52,50.34,17,24.40,55.79,62.26,57.80,16
1266,24.16,47.04,54.26,50.24,17,24.99,55.16,61.61,58.07,16


In [10]:
# Best model (loaded at the end of training): adapters + decision head only
model.save_pretrained(str(RUN / "best_adapter"))
tokenizer.save_pretrained(str(RUN / "best_adapter"))
print("saved", RUN / "best_adapter")

AttributeError: 'DecisionModel' object has no attribute 'save_pretrained'